In [1]:
%load_ext autoreload
%autoreload 2

### Qs 

- [ ] Normalizer ?
- [ ] opt chunk size ?
- [ ] opt use_bias ? 

In [53]:
import torch
import numpy as np

from hw1_imitation.model import MSEPolicy, build_policy
from hw1_imitation.train import TrainConfig, parse_train_config
from torch.utils.data import DataLoader

from hw1_imitation.data import (
    Normalizer,
    PushtChunkDataset,
    download_pusht,
    load_pusht_zarr,
)

In [33]:
config = TrainConfig(chunk_size=1, batch_size=1)
device = "cpu"

zarr_path = download_pusht(config.data_dir)
states, actions, episode_ends = load_pusht_zarr(zarr_path)
normalizer = Normalizer.from_data(states, actions)

dataset = PushtChunkDataset(
    states,
    actions,
    episode_ends,
    chunk_size=config.chunk_size,
    normalizer=normalizer,
)

loader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
)


print("State dim:", states.shape[1])
print("Action dim:", actions.shape[1])
print("Chunk size:", config.chunk_size)


model = build_policy(
    config.policy_type,
    state_dim=states.shape[1],
    action_dim=actions.shape[1],
    chunk_size=config.chunk_size,
    hidden_dims=config.hidden_dims,
    use_bias=config.use_bias,
).to(device)

print("model", model)

State dim: 5
Action dim: 2
Chunk size: 1
model MSEPolicy(
  (model): Sequential(
    (linear-0): Linear(in_features=5, out_features=256, bias=False)
    (relu-0): ReLU()
    (linear-1): Linear(in_features=256, out_features=256, bias=False)
    (relu-1): ReLU()
    (linear-2): Linear(in_features=256, out_features=256, bias=False)
    (relu-2): ReLU()
    (linear-3): Linear(in_features=256, out_features=2, bias=False)
  )
)


In [58]:
np.sqrt(64)

np.float64(8.0)

In [52]:
len(loader)

25650

In [34]:
data_iter = iter(loader)

In [35]:
x, y = next(data_iter)

In [48]:
for x, y in data_iter:
    pass

In [45]:
pred_action_chunk = model.model(x).reshape(config.batch_size, model.chunk_size, model.action_dim)
loss = torch.mean(torch.norm(y - pred_action_chunk, p=2)**2)

In [47]:
model.parameters()

<generator object Module.parameters at 0x162066ea0>

In [ ]:
# [BS, |S|], [BS, C, |A|]

# state (5):
# ...


# action (2):
# ...
